# Module 1: Market Data Collection

This notebook collects and stores raw historical data for the Vietnamese stock market.

**Two datasets are collected:**
- `market_data.parquet` — Daily OHLCV data for ~300 VN stocks (universe for stock selection)
- `vnindex_data.parquet` — VNINDEX daily index data (used as market benchmark and regime filter)

**Output:** Both files saved to `data/` in Parquet format for fast downstream loading.

> ⚠️ Re-running this notebook will **incrementally update** existing files, not overwrite them. Safe to run regularly to keep data current.

## 1.1 Imports and Project Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.append(project_root)

import src.data_collect as dc

## 1.2 Stock Universe — OHLCV Collection

We download daily Open, High, Low, Close, Volume data for all stocks in our universe.
Data is saved in batches of 5 tickers at a time to handle API rate limits gracefully.

In [ ]:
STOCK_FILE_PATH = '../data/market_data.parquet'
START_DATE      = '2016-01-01'
BATCH_SIZE      = 5
stock_symbols = dc.get_tags()
print(f'Universe size: {len(stock_symbols)} stocks')

dc.update_market_data(
    STOCK_FILE_PATH,
    stock_symbols,
    START_DATE,
    BATCH_SIZE
)

### Validation — Stock Data

In [ ]:
df_stocks = pd.read_parquet(STOCK_FILE_PATH)

print(f'Shape          : {df_stocks.shape}')
print(f'Date range     : {df_stocks["date"].min()} → {df_stocks["date"].max()}')
print(f'Unique tickers : {df_stocks["Symbol"].nunique()}')
print(f'Missing close  : {df_stocks["close"].isna().sum()}')
df_stocks.tail(5)

## 1.3 Market Benchmark — VNINDEX Collection

The VNINDEX is Vietnam's main market index, analogous to the S&P 500.
It serves two roles in this project:
1. **Benchmark** — compare strategy returns against the market
2. **Regime filter** — detect high-volatility market environments where rebalancing is skipped

In [ ]:
VNINDEX_FILE_PATH = '../data/vnindex_data.parquet'
START_DATE        = '2016-01-01'

dc.fetch_indicator_data(
    ind_symbol='VNINDEX',
    start_date=START_DATE,
    file_path=VNINDEX_FILE_PATH
)

### Validation — VNINDEX Data

In [ ]:
df_vnindex = pd.read_parquet(VNINDEX_FILE_PATH)

print(f'Shape      : {df_vnindex.shape}')
print(f'Date range : {df_vnindex["date"].min()} → {df_vnindex["date"].max()}')
print(f'Missing    : {df_vnindex.isna().sum().sum()}')
df_vnindex.tail(5)

## Summary

| Dataset | File | Coverage |
|---|---|---|
| Stock universe | `data/market_data.parquet` | ~300 stocks, 2016 → present |
| Market benchmark | `data/vnindex_data.parquet` | VNINDEX, 2016 → present |

**Next step →** `02_Feature_Engineering.ipynb` — build predictive features from raw OHLCV data.